In [ ]:
import numpy as np
import pandas as pd
from IPython.display import clear_output
import matplotlib.pyplot as plt
import random
import json

In [ ]:
def lv_step(params, variables, dt=1):
    """
    Performs one step of the inner Lotka–Volterra interaction.
    Based on the first step of the complex conflict transformation p.(16)

    params: (a, b, c, d, e, f) - model parameters controlling growth and interaction rates
    variables: (p, q) - current populations of prey and predator
    dt: time step size (default 1)

    Returns:
        (p1, p2) - normalized populations after one step (stochastic form)
        z - total population used for normalization
    """

    a, b, c, d, e, f = params

    # Inner Lotka-Volterra update
    p, q = variables

    P1 = p + p*dt*(a - b*q - c*p)
    P2 = q + q*dt*(-d + e*p - f*q)

    # Normalize to obtain stochastic vector
    z = P1 + P2
    p1, p2 = P1 / z, P2 / z

    return np.array([p1, p2]), z

def combinedModel(p0, r0, alpha, params, n, dt=1):
    """
    Simulates the full conflict model combining inner Lotka–Volterra
    dynamics with outer conflict interaction between two non-annihilating systems.

    Based on the complex conflict transformation p.(16-17)

    p0, r0: initial vectors for the two systems
    alpha: conflict interaction coefficient
    params: (a, b, c, d, e, f) - Lotka–Volterra parameters
    n: total simulation time
    dt: time step size (default 1)

    Returns:
        P, R, t - arrays of population vectors and time values
    """

    # Initialize time and population lists
    t = [0.0]
    P = [np.asarray(p0, dtype=float)] # population vector of system 1
    R = [np.asarray(r0, dtype=float)] # population vector of system 2

    while t[-1] < n:

        # Apply inner Lotka-Volterra step separately to both systems.
        p, z_p = lv_step(params, P[-1], dt)
        r, z_r = lv_step(params, R[-1], dt)

        # Apply outer conflict interaction
        denom = (1 - alpha * np.inner(p, r))
        p1 = p*(1 - alpha * r) / denom
        r1 = r*(1 - alpha * p) / denom

        # Convert back to non-normalized population vectors
        P1 = p1 * z_p
        R1 = r1 * z_r

        # Add results to end of lists
        P.append(P1)
        R.append(R1)
        t.append(t[-1] + dt)

    return np.array(P), np.array(R), np.array(t)

$t = n-M,\dots,n-1.$

In our case
$M = 1000$

$
\Delta P_0^{(t)} = P_0^{(t)}-P_0^{(t+1)},
$

$
\Delta P_1^{(t)} = P_1^{(t)}-P_1^{(t+1)},
$

$
S(\alpha)= \frac{1}{M-1}\sum_{t=n-M}^{n-1}\left|\Delta P_0^{(t)}\right|\;+\ \frac{1}{M-1}\sum_{t=n-M}^{n-1}\left|\Delta P_1^{(t)}\right|.
$

In [ ]:
def stability_score(alpha, params, start, n):

    p0, r0 = start

    P, R, t = combinedModel(p0, r0, alpha, params, n)

    diff_p = np.diff(P[-1000:, 0])
    diff_r = np.diff(P[-1000:, 1])

    score = np.mean(np.abs(diff_p)) + np.mean(np.abs(diff_r))

    return score

Let
$S(\alpha) = stability\ score$

$\varepsilon = bifurcation\ threshold$ (in our case 1e-17),

$\mathcal{V} = \{\alpha : S(\alpha) \in \mathbb{R}\}$

$(\alpha_k)_{k=1}^{N} =$ Sampled alphas -  arithmetic increasing sequence with difference $\delta$

Then:

$\text{bifurcation exists} \iff
\exists k \in \{2,\dots,N\} \text{ with }
\alpha_{k-1},\alpha_k \in \mathcal V
\quad\text{s.t.}\quad
\big(S(\alpha_{k-1})-\varepsilon\big)\big(S(\alpha_k)-\varepsilon\big)\le 0.
$

In [ ]:
def hasBifurcation(alphas, params, start, n, eps, i, y):
    prev_score = None
    prev_alpha = None

    for alpha in alphas:
        score = stability_score(alpha, params, start, n)
        clear_output(True)
        print(f"Epoch {i+1}")
        print(y)
        print(params)
        print(prev_score, score, alpha)
        if not np.isfinite(score):
            prev_score = None
            prev_alpha = None
            continue
        if prev_score is not None:
            crossed = (prev_score - eps) * (score - eps) <= 0
            if crossed:
                return 1
        prev_score = score
        prev_alpha = alpha

    return 0

build $\boldsymbol{\alpha} = (\alpha_k)_{k=1}^{N},$

$\alpha_k \in [-1,1]\setminus\{0\} \quad \forall k$

In [ ]:
def build_alphas(d_alpha, start, end):
    if start > end:
        start, end = end, start

    alphas = np.arange(start, end + d_alpha, d_alpha)
    return alphas[alphas != 0]

In [ ]:
def sample_params():
    a = random.uniform(0.0, 0.7)
    b = random.uniform(0.0, 0.09)
    c = random.uniform(0.0, 0.01)
    d = random.uniform(0.0, 0.05)
    e = random.uniform(0.0, 0.03)
    f = random.uniform(0.0, 0.07)

    params = [a, b, c, d, e, f]

    return params

In [ ]:
# Independent parameters
p0 = np.array([3, 10])
r0 = np.array([5, 20])
n = 100000
start = [p0, r0]

# Epsilon
eps = 1e-17

# Alphas
d_alpha = 0.01
alphas = build_alphas(d_alpha, -1, 1)

In [ ]:
# Note - Increase k for more samples
k = 2
# Uniformly samples k set of parameters
X = [sample_params() for _ in range(k)]

In [ ]:
y = []

In [ ]:
for i in range(k):
  params = X[i]
  result = hasBifurcation(alphas, params, start, n, eps, i, y)
  y.append(result)

Epoch 2
[0]
[0.4890980190239157, 0.008886849369107767, 0.0096155762210856, 0.01720784856390961, 0.015999514039189453, 0.01957250488944111]
0.0 1.0658141036401503e-14 -0.99


In [ ]:
print(X, y)

[[0.46138685834408505, 0.05814628524844329, 0.0008718737909092356, 0.012362636772532243, 0.0026413709764351668, 0.05151367548403453], [0.4890980190239157, 0.008886849369107767, 0.0096155762210856, 0.01720784856390961, 0.015999514039189453, 0.01957250488944111]] [0, 1]


In [ ]:
# Create dictionary containing X and y to then convert to json
data = {"X": X,
        "y": y}

In [ ]:
# Save data to json file
with open("bifurcation_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)